Load the tokenizer:

In [1]:
from transformers import AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, tgt_lang=None)

Prepare your training and validation data:

In [2]:
rename_dict = {"ja": "jpn_Jpan",
               "en": "eng_Latn",
               "zh-CN": "zho_Hans",
               "th": "tha_Thai",
               "ru": "rus_Cyrl",
               "zh-TW": "zho_Hant",
               "id": "ind_Latn",
               "ko": "kor_Hang",
               "pt": "por_Latn", }

In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv("all_files_merged.csv")
df.drop("source", axis=1, inplace=True)  # delete the "source" column
df.replace(np.nan, None, inplace=True)  # replace np.nan with None
df.rename(columns=rename_dict, inplace=True)  # rename the columns
df

/tmp/ipykernel_861/4117125360.py:4: DtypeWarning: Columns (0,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("all_files_merged.csv")


,jpn_Jpan,eng_Latn,zho_Hans,tha_Thai,rus_Cyrl,zho_Hant,ind_Latn,kor_Hang,por_Latn
0,（覚醒）,(awakened),（觉醒）,None,None,None,None,None,None
1,俗,(crude),一般,None,None,None,None,None,None
2,（大）,(high),（大）,None,None,None,None,None,None
3,視界（に浮かぶ）,(hovering in my) field of vision/(on my) statu...,（出现在）视野,None,None,None,None,None,None
4,（中）,(medium),（中）,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...
108807,None,End time for prizes:,领奖结束时间：,เวลาปิดรับของรางวัล:,None,领奖结束时间：,End time for prizes:,수령 종료 시간:,End time for prizes:
108808,None,consult,上门请教,ปรึกษาหารือ,None,上门请教,consult,가르침을 청하다,consult
108809,None,Learning,学习,เรียนรู้,None,學習,Learning,학습,Learning
108810,None,Permanently add attributes to all pets after e...,食用美食后为所有宠物永久增加属性,เพิ่มคุณสมบัติให้สัตว์เลี้ยงทุกตัวอย่างถาวรหลั...,None,食用美食後為所有寵物永久增加屬性,Permanently add attributes to all pets after e...,음식을 먹으면 모든 펫의 속성 영구 증가,Permanently add attributes to all pets after e...


In [4]:
language_list = df.columns.tolist()
# x: source language, y: target language
language_pairs = [[x, y] for x in ["zho_Hans"] for y in language_list if x != y]
# language_pairs = [[x, y] for index, x in enumerate(language_list) for y in language_list[index:] if x != y]
language_pairs

[['zho_Hans', 'jpn_Jpan'],
 ['zho_Hans', 'eng_Latn'],
 ['zho_Hans', 'tha_Thai'],
 ['zho_Hans', 'rus_Cyrl'],
 ['zho_Hans', 'zho_Hant'],
 ['zho_Hans', 'ind_Latn'],
 ['zho_Hans', 'kor_Hang'],
 ['zho_Hans', 'por_Latn']]

In [5]:
df_sub_list = []
for pair in language_pairs:
    df_sub = df[pair]
    df_sub = df_sub.dropna(how='any')  # clear all rows include nan
    df_sub = df_sub.drop_duplicates()  # delete exactly the same rows
    # filter the empty language pairs
    if len(df_sub) > 0:
        df_sub_list.append(df_sub)
len(df_sub_list)

8

find the max length of the encodings

In [6]:
from tqdm.notebook import tqdm

max_length = 0
for language in tqdm(language_list):
    texts = df[language].dropna(how='any').tolist()
    for text in texts:
        encodings = tokenizer(text)
        input_ids = encodings['input_ids']
        max_length = max(max_length, len(input_ids))
max_length

  0%|          | 0/9 [00:00<?, ?it/s]

1022

In [7]:
from tqdm.notebook import tqdm

train_encodings_list = []
for df_sub in tqdm(df_sub_list):
    languages = df_sub.columns.tolist()
    language_x, language_y = languages[0], languages[1]
    x_train = df_sub[language_x].to_list()
    y_train = df_sub[language_y].to_list()

    # add tokens not in the vocabulary of the tokenizer
    chars = list(set(''.join(x_train + y_train)))
    chars_not_in_vocab = [char for char in chars if 3 in tokenizer(char).input_ids]  # 3 is the value of unknown words
    tokenizer.add_tokens(chars_not_in_vocab)

    # Note: we're now creating separate encodings for the inputs and outputs.
    # truncation: truncate the sequence to a shorter length, because sometimes a sequence may be too long for a model to handle
    # padding: Padding is a strategy for ensuring tensors are rectangular by adding a special padding token to shorter sentences.
    #     True or 'longest': Pad to the longest sequence in the batch (or no padding if only a single sequence if provided).
    #     'max_length': Pad to a maximum length specified with the argument max_length or to the maximum acceptable input length for the model if that argument is not provided.
    #     False or 'do_not_pad' (default): No padding (i.e., can output a batch with sequences of different lengths).
    # return_tensors: If set 'pt', will return tensors instead of list of python integers. Acceptable values are PyTorch torch.Tensor objects.
    # max_length (int, optional): Controls the maximum length to use by one of the truncation/padding parameters.
    # 注意：一定要注意这个max_length的使用，当不同的批次要堆叠在一起时，不可以设置为True，而是应该设置为‘max_length'，这样它才能被填充/截断到同一个长度
    tokenizer.src_lang = language_x
    tokenizer.tgt_lang = language_y
    print("tokenizing - source:{}, target:{}".format(language_x, language_y))
    train_encodings = tokenizer(x_train, text_target=y_train, truncation=True, padding="max_length",
                                return_tensors="pt")

    train_encodings_list.append(train_encodings)
len(train_encodings_list)

  0%|          | 0/8 [00:00<?, ?it/s]

tokenizing - source:zho_Hans, target:jpn_Jpan
tokenizing - source:zho_Hans, target:eng_Latn
tokenizing - source:zho_Hans, target:tha_Thai
tokenizing - source:zho_Hans, target:rus_Cyrl
tokenizing - source:zho_Hans, target:zho_Hant
tokenizing - source:zho_Hans, target:ind_Latn
tokenizing - source:zho_Hans, target:kor_Hang
tokenizing - source:zho_Hans, target:por_Latn


8

Convert your encodings into torch Datasets object:

In [8]:
import torch


class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data_encoded_list: list):
        self.input_ids = []
        self.attention_mask = []
        self.labels = []
        for data_encoded in data_encoded_list:
            self.input_ids.extend(data_encoded.data["input_ids"])
            self.attention_mask.extend(data_encoded.data["attention_mask"])
            self.labels.extend(data_encoded.data["labels"])

    def __getitem__(self, index):
        item = {"input_ids": self.input_ids[index],
                "attention_mask": self.attention_mask[index],
                "labels": self.labels[index]}
        return item

    def __len__(self):
        return len(self.input_ids)


train_dataset = TranslationDataset(train_encodings_list)

Load the pretrained model:

In [9]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [10]:
# 调整embedding层的大小
model.resize_token_embeddings(len(tokenizer))

Embedding(258486, 1024)

Define your training arguments and train the model:

In [11]:
custom_model_name = "nllb-200/all"

In [12]:
from transformers import Trainer, TrainingArguments, IntervalStrategy

# fp16：半精度运算，启用后提高一倍以上运算速度，不影响loss
# gradient_accumulation_steps：steps越大，速度越快，loss越高
# gradient_checkpointing：启用后，降低30%左右速度，节省显存2/3
# per_device_train_batch_size：size越大，GPU占用率越大，速度越快，loss越高，几乎成正比
training_args = TrainingArguments(custom_model_name,
                                  num_train_epochs=1,
                                  per_device_eval_batch_size=1,
                                  per_device_train_batch_size=1,
                                  gradient_accumulation_steps=1,
                                  gradient_checkpointing=True,
                                  fp16=True,
                                  logging_strategy=IntervalStrategy.STEPS,
                                  logging_steps=2000,
                                  save_strategy=IntervalStrategy.STEPS,
                                  save_steps=2000,
                                  save_total_limit=1,
                                  )
from torch.utils import checkpoint  #未知的bug：不会自动加载这个包

In [13]:
trainer = Trainer(
    model=model,  # the instantiated 🤗 Transformers model to be trained
    args=training_args,  # training arguments, defined above
    train_dataset=train_dataset,  # training dataset
)

trainer.train(resume_from_checkpoint=False)

/root/miniconda3/lib/python3.8/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
2000,0.744400
4000,0.042600
6000,0.038300
8000,0.036100
10000,0.036200
12000,0.032300
14000,0.033800
16000,0.031100
18000,0.033100
20000,0.031100


TrainOutput(global_step=477496, training_loss=0.025692539159804274, metrics={'train_runtime': 109351.9608, 'train_samples_per_second': 4.367, 'train_steps_per_second': 4.367, 'total_flos': 1.0347837790900716e+18, 'train_loss': 0.025692539159804274, 'epoch': 1.0})

Save your fine-tuned model and tokenizer:

In [14]:
trainer.save_model(custom_model_name)
tokenizer.save_pretrained(custom_model_name)

('nllb-200/all/tokenizer_config.json',
 'nllb-200/all/special_tokens_map.json',
 'nllb-200/all/tokenizer.json')